<a href="https://colab.research.google.com/github/chamarairesh1982/ml-cybersecurity/blob/main/section2_intrusion/section2_intrusion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 2 — Network intrusion classification
**Independent learning lab • self-contained Colab notebook • development version 2**

Learn a complete pipeline for ten-category flow classification and a separate alert policy. Public data loads only in Colab. Python lives in this notebook; no local dataset, scripts, Drive mount or credentials are needed.

The previous saved run selected a forest but showed a substantial validation/test gap and exceeded its false-alert budget. Version 2 improves development methodology and candidate regularisation; improvement in independent real-data performance remains unmeasured. The old test is a historical benchmark, with replay disabled by default. A new compatible independent holdout is required before deployment claims.

## 1. Lecture concepts and scope
| Topic | Application |
|---|---|
| ML foundations | Target, prior baseline, generalisation and limitations |
| Data preparation / EDA | Types, invalid values, missingness, duplicate/conflicting records |
| Feature engineering | Flow ratios, log transforms, training-only imputation/scaling/encoding |
| Supervised learning | Regularised multiclass logistic regression |
| Neural networks / deep learning | Dense MLP, regularisation, dropout, early stopping, class weights |
| Ensemble learning | Random forest and histogram gradient boosting |
| Evaluation | Grouped CV, fixed partitions, macro/per-class metrics, confusion/PR, alert workload |
| Unsupervised learning | Reserved primarily for Section 3 |
| NLP | Not required for structured flow measurements |

The objective is strong **evidence**, not a predetermined winning algorithm. A CNN over arbitrary columns would impose an unsupported neighbourhood structure. Full-flow features are available only after measurement; this notebook does not establish packet-time detection or prevention.

## 2. Runtime and configuration
Classical scikit-learn training uses CPU. The numerical TensorFlow MLP supports CPU, GPU and TPU strategies; actual accelerator compatibility must be checked in your Colab runtime. Colab supplies major packages. The first cell installs only the download/parquet extras. Do not replace a managed TensorFlow installation indiscriminately.

The default uses all cleaned training rows, a bounded three-fold CV search, and two neural seeds. It may take substantial CPU time. Changing settings after examining the final test needs a fresh independent holdout.

In [ ]:
%pip -q install "requests>=2.31,<3" "pyarrow>=14"

In [ ]:
import os, io, re, json, time, random, hashlib, platform
from pathlib import Path
import importlib.metadata as metadata
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay)
from sklearn.inspection import permutation_importance
from sklearn.metrics import make_scorer
from scipy.stats import binom
from joblib import parallel_backend
from threadpoolctl import threadpool_limits

SEED = 42
random.seed(SEED); np.random.seed(SEED)
EXPERIMENT_VERSION = "section2-v2-development"
RUN_HISTORICAL_TEST = False  # Already inspected benchmark; never a fresh holdout.
POLICY_CONFIDENCE = .95
RUN_NEURAL = True
ACCELERATOR = "auto"  # auto, cpu, gpu, tpu
N_JOBS = 2
CPU_THREADS_PER_WORKER = 2  # Avoid nested BLAS/OpenMP oversubscription.
CV_FOLDS = 3
EPOCHS = 20
BOOTSTRAPS = 200
TARGET_FPR = .01  # Illustrative validation budget on Normal-labelled flows.
RUN_PERMUTATION_IMPORTANCE = False  # Validation-only diagnostic; potentially expensive.
CLASSES = ["Normal","Analysis","Backdoor","DoS","Exploits","Fuzzers",
           "Generic","Reconnaissance","Shellcode","Worms"]
CLASS_TO_ID = {name:i for i,name in enumerate(CLASSES)}
NORMAL_ID = CLASS_TO_ID["Normal"]
CATEGORICAL = ["proto","service","state"]
RAW_NUMERIC = ["dur","spkts","dpkts","sbytes","dbytes","rate","sttl","dttl",
    "sload","dload","sloss","dloss","sinpkt","dinpkt","sjit","djit","swin",
    "stcpb","dwin","dtcpb","tcprtt","synack","ackdat","smean","dmean","trans_depth",
    "response_body_len","ct_srv_src","ct_state_ttl","ct_dst_ltm","ct_src_dport_ltm",
    "ct_dst_sport_ltm","ct_dst_src_ltm","is_ftp_login","ct_ftp_cmd","ct_flw_http_mthd",
    "ct_src_ltm","ct_srv_dst","is_sm_ips_ports"]
# TCP initial sequence numbers can behave as collection-specific identifiers.
NUMERIC = [c for c in RAW_NUMERIC if c not in {"stcpb","dtcpb"}]
PREDICTORS = NUMERIC + CATEGORICAL
DERIVED = ["src_bytes_per_packet","dst_bytes_per_packet",
           "byte_direction_ratio","packet_direction_ratio"]
CORE_NUMERIC = [c for c in NUMERIC if not c.startswith("ct_") and c not in {"sttl","dttl"}]
REVISION = "34ee5744ecb143db1b12623670f754f56c36b94b"
BASE_URL = f"https://huggingface.co/datasets/lacg030175/UNSW-NB15/resolve/{REVISION}"
FILES = {
    "train":("standard/train-00000-of-00001.parquet",
             "0c4a4884abfe24d4545c5ed8cbe1146525cc4ac5c8e0dc8c2b654309a32616ed"),
    "test":("standard/test-00000-of-00001.parquet",
            "70292cae81a8f24aa89ed6c74b01686db36b0750ef9af94da7cdc54e166be598")}
versions = {p:metadata.version(p) for p in ["numpy","pandas","scikit-learn","matplotlib","pyarrow","scipy"]}
print("Python:",platform.python_version(),"\nPackages:",versions)
print("Frozen target categories:",CLASSES)

## 3. Provenance, integrity and benchmark limits
Use the [original UNSW-NB15 description](https://research.unsw.edu.au/projects/unsw-nb15-dataset) and the [pinned mirror card](https://huggingface.co/datasets/lacg030175/UNSW-NB15/blob/34ee5744ecb143db1b12623670f754f56c36b94b/README.md). The original provider describes hybrid normal activity and synthetic attacks, nine attack categories, and published training/test partitions. This mirror's `standard` configuration reformats those partitions into parquet.

Original terms permit academic research and require attribution; commercial use requires agreement. The mirror states a broader CC-BY licence, which this notebook does **not** assume overrides original rights. No data are redistributed here.

The mirror calls this temporal, but the reformatted partition lacks reliable timestamps/session IDs to independently verify chronology. Treat it as the **publisher benchmark split**, not proved future-traffic validation. Do not combine train/test into a favourable random split. Full feature identity and rounded feature groups are only proxies for related flows, not verified session grouping.

The benchmark permits corpus-level learning comparisons. Packet/flow statistics and contextual `ct_*` features may require completed flows/history; availability in a real detector must be established. A declared core-feature ablation excludes TTL/context counts to investigate reliance on collection-specific signals.

In [ ]:
def load_partition(name):
    relative_path, expected = FILES[name]
    response = requests.get(f"{BASE_URL}/{relative_path}",timeout=(20,180))
    response.raise_for_status()
    payload = response.content
    digest = hashlib.sha256(payload).hexdigest()
    if digest != expected:
        raise ValueError(f"{name} checksum mismatch; investigate before changing the pin.")
    frame = pd.read_parquet(io.BytesIO(payload))
    return frame, {"sha256":digest,"bytes":len(payload),"rows":len(frame)}

# Test is deliberately not loaded here. It is fetched only after validation decisions freeze.
raw_train, download_train = load_partition("train")
print("Training partition loaded into Colab memory:",download_train)

## 4. Raw training audit and deterministic cleaning
Inspect types, missingness, categorical cardinality and target counts. Never use `id`, `label`, `attack_cat` or partition identity as predictors. Validate consistency between category and binary labels. Numeric invalid/infinite/negative values become missing and are counted; do not delete high traffic values merely as statistical outliers.

Normalise category spelling (including Backdoors → Backdoor). Empty categorical fields become missing; the service `-` marker is retained as a meaningful 'no named service' category. Missing labels and contradictory feature-identical targets are excluded with counts.

Hash only the actual base predictor representation. Retain one consistent copy of feature-identical rows. Scores on deduplicated, overlap-filtered test rows will differ from an unmodified row-weighted published benchmark, so explicitly report the evaluation population.

In [ ]:
def clean_partition(frame):
    expected = set(RAW_NUMERIC + CATEGORICAL + ["attack_cat","label"])
    if not expected.issubset(frame.columns):
        raise ValueError(f"Missing fields: {sorted(expected - set(frame.columns))}")
    unexpected = set(frame.columns)-expected-{"id","__index_level_0__"}
    if unexpected:
        raise ValueError(f"Unexpected columns need explicit review: {sorted(unexpected)}")
    audit = {"input_rows":len(frame)}
    work = frame.copy()
    category = work["attack_cat"].astype("string").str.strip().replace({"Backdoors":"Backdoor"})
    missing_label = category.isna() | category.eq("") | work["label"].isna()
    audit["missing_target_rows_removed"] = int(missing_label.sum())
    work = work.loc[~missing_label].copy()
    category = category.loc[~missing_label]
    unknown = set(category.unique())-set(CLASSES)
    if unknown:
        raise ValueError(f"Unexpected target categories: {unknown}")
    binary = pd.to_numeric(work["label"],errors="coerce")
    if not binary.isin([0,1]).all():
        raise ValueError("Binary targets must be 0 or 1.")
    if not ((category != "Normal").astype(int).to_numpy() == binary.to_numpy()).all():
        raise ValueError("Category/binary-label inconsistency; investigate labels.")
    work["target"] = category.map(CLASS_TO_ID).astype("int32")
    invalid_counts = {}
    for column in RAW_NUMERIC:
        original = work[column]
        values = pd.to_numeric(original,errors="coerce").astype("float64")
        invalid = ~np.isfinite(values) | (values < 0)
        invalid_counts[column] = int((invalid & original.notna()).sum())
        work[column] = values.mask(invalid,np.nan)
    for column in CATEGORICAL:
        values = work[column].astype("string").str.strip().str.lower()
        work[column] = values.mask(values.eq(""),pd.NA).astype(object)
        work[column] = work[column].where(pd.notna(work[column]),np.nan)
    # index=False excludes row identity; a 64-bit feature hash is a practical duplicate proxy.
    work["feature_hash"] = pd.util.hash_pandas_object(work[PREDICTORS],index=False).astype("uint64")
    clashes = work.groupby("feature_hash")["target"].nunique()
    conflicted = work["feature_hash"].isin(clashes[clashes>1].index)
    audit["conflicting_feature_duplicate_rows_removed"] = int(conflicted.sum())
    work = work.loc[~conflicted].copy()
    audit["consistent_feature_duplicate_rows_removed"] = int(work["feature_hash"].duplicated().sum())
    work = work.drop_duplicates("feature_hash").reset_index(drop=True)
    audit["retained_rows"] = len(work)
    input_support = category.value_counts().reindex(CLASSES,fill_value=0)
    retained_support = work["target"].value_counts().reindex(range(len(CLASSES)),fill_value=0)
    audit["class_retention"] = {name:{"input_labelled_rows":int(input_support[name]),
        "retained_rows":int(retained_support[i]),
        "excluded_rows":int(input_support[name]-retained_support[i])}
        for i,name in enumerate(CLASSES)}
    audit["invalid_numeric_values_to_missing"] = invalid_counts
    return work[PREDICTORS+["target","feature_hash"]], audit

display(pd.DataFrame({"dtype":raw_train.dtypes.astype(str),"missing":raw_train.isna().sum()}))
display(raw_train["attack_cat"].value_counts(dropna=False).to_frame("training_target_count"))
print("Raw training shape:",raw_train.shape)
train_pool, cleaning_train = clean_partition(raw_train)
del raw_train
display(pd.Series({k:v for k,v in cleaning_train.items() if isinstance(v,int)}).to_frame("count"))
display(pd.Series(cleaning_train["invalid_numeric_values_to_missing"]).to_frame("invalid_values"))
display(pd.DataFrame.from_dict(cleaning_train["class_retention"],orient="index"))
print("Label-dependent duplicate exclusions change the evaluation population; inspect rare-class retention.")


## 5. Separate fitting, model selection and policy calibration
Use the same approximate rounded core-feature groups as the original experiment. Groups remain disjoint in three fixed development partitions: about 60% fitting, 20% model selection and neural stopping, and 20% alert-policy calibration. Grouped CV stays inside fitting data. No seed search, resampling outside folds or test-driven split selection.

This reduces threshold/model selection reuse. Partition support is printed because rare classes need explicit scrutiny. These are development partitions from a previously used training corpus, not new external holdouts. Rounded feature groups do not prove source/session independence.

In [ ]:
def rounded_feature_groups(frame):
    rounded = frame[CORE_NUMERIC + CATEGORICAL].copy()
    for column in CORE_NUMERIC:
        values = rounded[column].to_numpy(dtype="float64")
        nonzero = np.isfinite(values) & (values != 0)
        precision = np.ones_like(values)
        precision[nonzero] = np.power(10.0,np.clip(3-np.floor(np.log10(np.abs(values[nonzero]))),-300,300))
        rounded[column] = np.round(values*precision)/precision
    return pd.util.hash_pandas_object(rounded,index=False).to_numpy(dtype="uint64")

train_pool["group"] = rounded_feature_groups(train_pool)
group_counts = train_pool.groupby("target")["group"].nunique()
display(group_counts.rename(index=dict(enumerate(CLASSES))).to_frame("independent_feature_groups"))
if set(group_counts.index) != set(range(len(CLASSES))) or group_counts.min() < 10:
    raise ValueError("Every target needs at least ten independent feature groups.")
# Declare folds once; never search for a lucky split.
splitter = StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=SEED)
folds = [held for _,held in splitter.split(train_pool,train_pool["target"],train_pool["group"])]
training = train_pool.iloc[np.concatenate(folds[:3])].reset_index(drop=True)
validation = train_pool.iloc[folds[3]].reset_index(drop=True)  # model selection / neural stopping
policy = train_pool.iloc[folds[4]].reset_index(drop=True)  # only after winner freezes
development_parts = {"training":training,"model_selection":validation,"policy_calibration":policy}
for name,frame in development_parts.items():
    if set(frame["target"]) != set(range(len(CLASSES))):
        raise ValueError(f"{name} lacks a category; inspect data, do not try lucky seeds.")
for a,b in [("training","model_selection"),("training","policy_calibration"),("model_selection","policy_calibration")]:
    assert not set(development_parts[a]["group"]) & set(development_parts[b]["group"])
    assert not set(development_parts[a]["feature_hash"]) & set(development_parts[b]["feature_hash"])
X_train,y_train = training[PREDICTORS],training["target"].to_numpy()
X_val,y_val = validation[PREDICTORS],validation["target"].to_numpy()
g_train = training["group"].to_numpy()
cv = StratifiedGroupKFold(n_splits=CV_FOLDS,shuffle=True,random_state=SEED+1)
cv_splits = list(cv.split(X_train,y_train,g_train))
for tr,va in cv_splits:
    assert not set(g_train[tr]) & set(g_train[va])
    if set(y_train[tr])!=set(range(len(CLASSES))) or set(y_train[va])!=set(range(len(CLASSES))):
        raise ValueError("A CV fold lacks a target category.")
display(pd.DataFrame([{"partition":name,"rows":len(frame),"groups":frame["group"].nunique()}
                     for name,frame in development_parts.items()]))
display(pd.DataFrame({name:frame["target"].value_counts().reindex(range(len(CLASSES)),fill_value=0)
                     for name,frame in development_parts.items()}).rename(index=dict(enumerate(CLASSES))))
print("Policy predictors/scores stay unused until model selection freezes; historical test remains unfetched.")


## 6. Training-only EDA and availability review
Plot class support, missingness and duration. Inspect skew rather than deleting rare attacks or large values. Class weights are tuned inside training; no resampling before splitting. Do not convert rare attacks into Normal.

Some learned predictors may be dataset shortcuts: inspect TTL and contextual counts, use the core-feature comparison, and acknowledge missing independent deployment data.

In [ ]:
fig,axes = plt.subplots(1,2,figsize=(13,4))
training["target"].value_counts().reindex(range(len(CLASSES)),fill_value=0).plot.bar(ax=axes[0])
axes[0].set_xticklabels(CLASSES,rotation=60,ha="right"); axes[0].set_title("Training class support")
axes[1].hist(np.log1p(training["dur"].dropna()),bins=50)
axes[1].set_xlabel("log(1 + duration)"); axes[1].set_title("Training duration")
plt.tight_layout(); plt.show()
display(X_train.isna().mean().sort_values(ascending=False).head(10).to_frame("missing_fraction"))
display(X_train[CATEGORICAL].nunique().to_frame("training_cardinality"))
print("Normal fraction:",float(np.mean(y_train==NORMAL_ID)))

## 7. Feature engineering and leakage-safe pipelines
Create directional ratios and bytes per packet using only row-local measurements; zero denominators produce missing values, then a fitted imputer handles them. Preserve all missing indicators.

Numeric pipeline: median imputation → nonnegative log1p → standardisation for linear/neural models. Tree models use imputation/log transforms without scaling. One-hot encode protocol/service/state, treating unknown categories safely. Rare-category grouping is learned only from each fit. Dense float32 matrices are bounded by feature counts; never fit the encoder or imputer outside training.

The core-feature boosted model excludes TTL and contextual count features. This is a predeclared availability/artifact ablation, not a claim that the remaining features are causally valid.

In [ ]:
class FlowRatios(BaseEstimator,TransformerMixin):
    def fit(self,X,y=None):
        return self
    def transform(self,X):
        frame=X.copy()
        def divide(numerator,denominator):
            a=frame[numerator].to_numpy(dtype="float64")
            b=frame[denominator].to_numpy(dtype="float64")
            out=np.full(len(frame),np.nan)
            with np.errstate(over="ignore",invalid="ignore"):
                np.divide(a,b,out=out,where=np.isfinite(b)&(b>0))
            out[~np.isfinite(out)] = np.nan
            return out
        frame["src_bytes_per_packet"]=divide("sbytes","spkts")
        frame["dst_bytes_per_packet"]=divide("dbytes","dpkts")
        frame["byte_direction_ratio"]=divide("sbytes","dbytes")
        frame["packet_direction_ratio"]=divide("spkts","dpkts")
        return frame

def log_numeric(values):
    return np.log1p(values).astype("float32")

def preprocessing(scale=False,core=False):
    numeric_columns=(CORE_NUMERIC if core else NUMERIC)+DERIVED
    steps=[("impute",SimpleImputer(strategy="median",add_indicator=True,keep_empty_features=True)),
           ("log",FunctionTransformer(log_numeric,feature_names_out="one-to-one"))]
    if scale:
        steps.append(("scale",StandardScaler()))
    categorical=Pipeline([
        ("impute",SimpleImputer(strategy="constant",fill_value="missing")),
        ("encode",OneHotEncoder(handle_unknown="infrequent_if_exist",
            min_frequency=10,sparse_output=False,dtype=np.float32))])
    transform=ColumnTransformer([
        ("numeric",Pipeline(steps),numeric_columns),
        ("categorical",categorical,CATEGORICAL)],sparse_threshold=0)
    return Pipeline([("ratios",FlowRatios()),("columns",transform)])

def candidate(model,scale=False,core=False):
    return Pipeline([("prepare",preprocessing(scale,core)),("model",model)])

experiments={
    "logistic":(candidate(LogisticRegression(max_iter=1500,random_state=SEED),scale=True),
        {"model__C":[.3,1.0,3.0],"model__class_weight":[None,"balanced"]}),
    "forest":(candidate(RandomForestClassifier(n_estimators=200,
                    n_jobs=1,random_state=SEED)),
        {"model__max_depth":[16,None],"model__min_samples_leaf":[4,12],
         "model__class_weight":[None,"balanced"]}),
    "extra_trees":(candidate(ExtraTreesClassifier(n_estimators=200,class_weight="balanced",
                    n_jobs=1,random_state=SEED)),
        {"model__min_samples_leaf":[2,8],"model__max_features":[.5,1.0]}),
    "boosting":(candidate(HistGradientBoostingClassifier(max_iter=100,
                  learning_rate=.08,early_stopping=False,class_weight="balanced",random_state=SEED)),
        {"model__max_leaf_nodes":[15,31],"model__l2_regularization":[1.0,10.0]}),
    "boosting_core":(candidate(HistGradientBoostingClassifier(max_iter=100,
                   learning_rate=.08,early_stopping=False,class_weight="balanced",
                   random_state=SEED),core=True),
        {"model__max_leaf_nodes":[15,31]})
}
print("Declared candidates:",list(experiments))
print("Histogram boosting internal early stopping is disabled; grouped outer CV selects settings.")

## 8. Grouped CV, regularisation and model comparison
Compare prior, logistic regression, regularised forest, Extra Trees, histogram boosting and a core-feature ablation. The randomized-tree ensemble is a candidate, not a promised winner. Bounded grids include leaf-size regularisation and boosting L2 regularisation. Do not apply SMOTE or fit encoders before splitting.

Hyperparameters maximise grouped-CV mean macro-F1 minus its fold standard deviation, a declared stability heuristic rather than a statistical confidence bound. Print train/CV gaps and macro recall. Limit dispatched jobs to avoid excess matrix copies. All classical candidates use CPU. [Extra Trees documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesClassifier.html).

In [ ]:
def stable_refit(results):
    # Heuristic stability penalty, not a confidence bound.
    utility = np.asarray(results["mean_test_macro_F1"]) - np.asarray(results["std_test_macro_F1"])
    if not np.isfinite(utility).any():
        raise ValueError("No finite cross-validation scores.")
    return int(np.nanargmax(utility))

scoring = {
    "macro_F1":make_scorer(f1_score,labels=np.arange(len(CLASSES)),average="macro",zero_division=0),
    "macro_recall":make_scorer(recall_score,labels=np.arange(len(CLASSES)),average="macro",zero_division=0)}
models,validation_probabilities,fit_seconds,cv_records={},{},{},[]
baseline=DummyClassifier(strategy="prior").fit(np.zeros((len(y_train),1)),y_train)
models["prior"]=baseline
validation_probabilities["prior"]=baseline.predict_proba(np.zeros((len(y_val),1)))
fit_seconds["prior"]=0.0
for name,(pipeline,grid) in experiments.items():
    print("Fitting grouped search:",name)
    start=time.perf_counter()
    search=GridSearchCV(pipeline,grid,scoring=scoring,cv=cv_splits,
        n_jobs=N_JOBS,pre_dispatch=N_JOBS,refit=stable_refit,
        return_train_score=True,error_score="raise")
    with parallel_backend("loky",inner_max_num_threads=CPU_THREADS_PER_WORKER), threadpool_limits(limits=CPU_THREADS_PER_WORKER):
        search.fit(X_train,y_train)
    fit_seconds[name]=time.perf_counter()-start
    models[name]=search.best_estimator_
    assert np.array_equal(models[name].classes_,np.arange(len(CLASSES)))
    with threadpool_limits(limits=CPU_THREADS_PER_WORKER):
        validation_probabilities[name]=models[name].predict_proba(X_val)
    best=search.best_index_
    mean=search.cv_results_["mean_test_macro_F1"][best]
    std=search.cv_results_["std_test_macro_F1"][best]
    train_score=search.cv_results_["mean_train_macro_F1"][best]
    cv_records.append({"model":name,"CV_macro_F1":mean,"CV_std":std,
        "CV_stability_utility":mean-std,"train_macro_F1":train_score,
        "train_CV_gap":train_score-mean,
        "CV_macro_recall":search.cv_results_["mean_test_macro_recall"][best],
        "search_seconds":fit_seconds[name],"parameters":search.best_params_})
cv_table=pd.DataFrame(cv_records).sort_values("CV_stability_utility",ascending=False)
display(cv_table)


## 9. Numerical neural comparator
Fit imputation, encoding and scaling only on fitting data. Use square-root inverse-frequency weights, normalized to average sample weight one, to temper the influence of tiny classes. The validation callback records fixed-taxonomy macro-F1 and early stopping restores the best macro-F1 epoch. Model-selection data remain development data, also used for neural stopping.

Run two declared seeds, regularisation and dropout. Preprocessing stays on CPU; only numerical batches enter TensorFlow GPU/TPU strategies. Callback prediction adds some epoch overhead. [TensorFlow callback guide](https://www.tensorflow.org/guide/keras/writing_your_own_callbacks).

In [ ]:
if RUN_NEURAL:
    import tensorflow as tf
    versions["tensorflow"]=tf.__version__
    if ACCELERATOR not in {"auto","cpu","gpu","tpu"}:
        raise ValueError("Unknown ACCELERATOR.")
    strategy=None
    if ACCELERATOR in {"auto","tpu"}:
        try:
            resolver=tf.distribute.cluster_resolver.TPUClusterResolver()
        except ValueError:
            if ACCELERATOR=="tpu":
                raise RuntimeError("TPU requested but unavailable. Select a TPU Colab runtime.")
        else:
            tf.config.experimental_connect_to_cluster(resolver)
            tf.tpu.experimental.initialize_tpu_system(resolver)
            strategy=tf.distribute.TPUStrategy(resolver)
    if strategy is None:
        devices=tf.config.list_logical_devices("GPU")
        if ACCELERATOR=="gpu" and not devices:
            raise RuntimeError("GPU requested but unavailable.")
        strategy=(tf.distribute.MirroredStrategy() if devices and ACCELERATOR!="cpu"
                  else tf.distribute.OneDeviceStrategy("/CPU:0"))
    print("Strategy:",type(strategy).__name__,"replicas:",strategy.num_replicas_in_sync)
    neural_preprocessor=preprocessing(scale=True)
    train_matrix=neural_preprocessor.fit_transform(X_train,y_train).astype("float32")
    val_matrix=neural_preprocessor.transform(X_val).astype("float32")
    if not np.isfinite(train_matrix).all() or not np.isfinite(val_matrix).all():
        raise ValueError("Neural matrix contains nonfinite values.")
    print("Numeric input width:",train_matrix.shape[1],
          "training MB:",train_matrix.nbytes/1e6)
    GLOBAL_BATCH=64*strategy.num_replicas_in_sync
    def neural_dataset(matrix,labels=None,training=False):
        ds=(tf.data.Dataset.from_tensor_slices(matrix) if labels is None
            else tf.data.Dataset.from_tensor_slices((matrix,labels.astype("int32"))))
        if training:
            ds=ds.shuffle(len(matrix),seed=SEED,reshuffle_each_iteration=True)
            ds=ds.repeat().take(int(np.ceil(len(matrix)/GLOBAL_BATCH))*GLOBAL_BATCH)
        return ds.batch(GLOBAL_BATCH,drop_remainder=training).prefetch(tf.data.AUTOTUNE)

    class_counts=np.bincount(y_train,minlength=len(CLASSES))
    # Temper rare-class weights rather than giving tiny classes extreme gradients.
    raw_weights=np.sqrt(len(y_train)/(len(CLASSES)*class_counts))
    raw_weights=raw_weights/(np.dot(raw_weights,class_counts)/len(y_train))
    weights={i:float(w) for i,w in enumerate(raw_weights)}
    class MacroF1Monitor(tf.keras.callbacks.Callback):
        def on_epoch_end(self,epoch,logs=None):
            probabilities=self.model.predict(neural_dataset(val_matrix),verbose=0)
            value=f1_score(y_val,probabilities.argmax(axis=1),
                labels=np.arange(len(CLASSES)),average="macro",zero_division=0)
            if logs is not None:
                logs["val_macro_F1"]=float(value)
            print(f" — validation macro-F1: {value:.4f}")
    neural_histories={}
    for seed in [SEED,SEED+7]:
        tf.keras.utils.set_random_seed(seed)
        with strategy.scope():
            network=tf.keras.Sequential([
                tf.keras.Input(shape=(train_matrix.shape[1],),dtype="float32"),
                tf.keras.layers.Dense(128,activation="relu",
                    kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
                tf.keras.layers.Dropout(.3),
                tf.keras.layers.Dense(64,activation="relu",
                    kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
                tf.keras.layers.Dropout(.2),
                tf.keras.layers.Dense(len(CLASSES),activation="softmax")])
            network.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                loss="sparse_categorical_crossentropy",metrics=["accuracy"])
        start=time.perf_counter()
        history=network.fit(neural_dataset(train_matrix,y_train,True),
            validation_data=neural_dataset(val_matrix,y_val),epochs=EPOCHS,
            class_weight=weights,shuffle=False,verbose=2,
            callbacks=[MacroF1Monitor(),tf.keras.callbacks.EarlyStopping(
                monitor="val_macro_F1",mode="max",min_delta=1e-3,patience=3,restore_best_weights=True)])
        name=f"mlp_seed_{seed}"
        models[name]=network
        fit_seconds[name]=time.perf_counter()-start
        validation_probabilities[name]=network.predict(neural_dataset(val_matrix),verbose=0)
        neural_histories[name]=history.history
    fig,axes=plt.subplots(1,2,figsize=(11,4))
    for name,h in neural_histories.items():
        axes[0].plot(h["loss"],label=name+" train")
        axes[0].plot(h["val_loss"],"--",label=name+" validation")
        axes[1].plot(h["val_macro_F1"],label=name)
    axes[0].set_title("Neural loss");axes[1].set_title("Validation macro-F1")
    for ax in axes:ax.set_xlabel("Epoch index");ax.legend(fontsize=7)
    plt.tight_layout();plt.show()
else:
    print("Neural comparator disabled; classical experiment remains available.")

## 10. Freeze the model, then calibrate the alert policy
Select the category classifier on model-selection macro-F1, then Normal-category FPR and fitting cost. Print every class's precision, recall and support. Freeze the winner **before** using the separate policy partition. Do not choose a different model from policy measurements.

For each predeclared model, calibrate a conservative alert threshold on Normal scores only. Collapse related-feature groups to their maximum Normal score. Choose an order-statistic rank using a binomial tail criterion at the declared target and confidence; exclude ties. If there are too few groups, reject all alerts and explicitly report that limitation. The default targets a 1% false-alert probability for a group containing Normal flows at 95% confidence **under independent, exchangeable group-score assumptions**. This is not a guarantee for row-weighted FPR or a changed deployment population. Approximate groups may remain dependent; these assumptions need independent validation.

Category argmax and thresholded attack alerts are different decisions. A class-weighted score is not assumed calibrated probability. Threshold selection does not use policy attack labels; recall is measured afterward and may fall substantially. Individual-model bounds are not simultaneous guarantees across all models. [SciPy binomial distribution](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.binom.html).

PSI uses fitting-only numeric quantile bins to describe policy covariate differences; it does not prove drift causality or set a deployment acceptance rule.

In [ ]:
def check_probabilities(probabilities,n):
    probabilities=np.asarray(probabilities)
    if probabilities.shape!=(n,len(CLASSES)) or not np.isfinite(probabilities).all():
        raise ValueError("Invalid category probability matrix.")
    if (probabilities<0).any() or (probabilities>1).any() or not np.allclose(probabilities.sum(axis=1),1,atol=1e-5):
        raise ValueError("Probabilities must be normalized and in [0,1].")
    return probabilities

def predict_category(name,frame):
    if name=="prior":
        pp=models[name].predict_proba(np.zeros((len(frame),1)))
    elif name.startswith("mlp_seed"):
        matrix=neural_preprocessor.transform(frame).astype("float32")
        pp=models[name].predict(neural_dataset(matrix),verbose=0)
    else:
        with threadpool_limits(limits=CPU_THREADS_PER_WORKER):
            pp=models[name].predict_proba(frame)
    return check_probabilities(pp,len(frame))

# Freeze the category winner before accessing policy predictors or scores.
selection_rows=[]
for name,pp in validation_probabilities.items():
    pp=check_probabilities(pp,len(y_val))
    predicted=pp.argmax(axis=1)
    selection_rows.append({"model":name,
        "validation_macro_F1":f1_score(y_val,predicted,labels=np.arange(len(CLASSES)),average="macro",zero_division=0),
        "validation_macro_recall":recall_score(y_val,predicted,labels=np.arange(len(CLASSES)),average="macro",zero_division=0),
        "normal_category_FPR":float(np.mean(predicted[y_val==NORMAL_ID]!=NORMAL_ID)),
        "fit_seconds":fit_seconds[name]})
selection_table=pd.DataFrame(selection_rows).sort_values(
    ["validation_macro_F1","normal_category_FPR","fit_seconds"],ascending=[False,True,True])
winner=selection_table.iloc[0]["model"]
display(selection_table)
print("Frozen category winner:",winner)
selection_per_class=pd.DataFrame(classification_report(y_val,
    validation_probabilities[winner].argmax(axis=1),labels=np.arange(len(CLASSES)),
    target_names=CLASSES,output_dict=True,zero_division=0)).T
display(selection_per_class)

def conservative_group_threshold(normal_group_scores,max_fpr,confidence):
    scores=np.asarray(normal_group_scores,dtype="float64")
    if scores.ndim!=1 or not np.isfinite(scores).all() or (scores<0).any() or (scores>1).any():
        raise ValueError("Normal group scores must be finite probabilities.")
    if not 0<max_fpr<1 or not 0<confidence<1:
        raise ValueError("Invalid policy settings.")
    n=len(scores)
    # Choose rank in advance using only n, target risk and confidence.
    # P(population tail exceeds target) <= BinomialCDF(k;n,target)
    # for this order statistic under independent exchangeable group scores.
    feasible=np.flatnonzero(binom.cdf(np.arange(n),n,max_fpr)<=1-confidence)
    if not len(feasible):
        return float(np.nextafter(1.0,np.inf)),{
            "normal_groups":n,"allowed_exceedances":0,
            "rank_bound_failure_probability":0.0,
            "policy_status":"insufficient_groups_reject_all"}
    k=int(feasible[-1])
    ordered=np.sort(scores)[::-1]
    threshold=float(np.nextafter(ordered[k],np.inf))  # ties excluded conservatively
    return threshold,{"normal_groups":n,"allowed_exceedances":k,
        "rank_bound_failure_probability":float(binom.cdf(k,n,max_fpr)),
        "policy_status":"group_order_statistic"}

X_policy,y_policy=policy[PREDICTORS],policy["target"].to_numpy()
normal_policy=y_policy==NORMAL_ID
binary_policy=(~normal_policy).astype(int)
thresholds,policy_rows,policy_probabilities,policy_details={},[],{},{}
for name in models:
    pp=predict_category(name,X_policy)
    policy_probabilities[name]=pp
    scores=1-pp[:,NORMAL_ID]
    # One maximum over Normal-labelled flows per related-feature group.
    group_scores=pd.Series(scores[normal_policy]).groupby(
        policy.loc[normal_policy,"group"].to_numpy()).max().to_numpy()
    thresholds[name],details=conservative_group_threshold(group_scores,TARGET_FPR,POLICY_CONFIDENCE)
    policy_details[name]=details
    alerts=scores>=thresholds[name]
    policy_rows.append({"model":name,"selected_on_model_selection":name==winner,
        "threshold":thresholds[name],
        "policy_alert_recall":recall_score(binary_policy,alerts,zero_division=0),
        "policy_row_FPR":float(np.mean(alerts[normal_policy])),
        "policy_normal_group_FPR":float(np.mean(group_scores>=thresholds[name])),
        **details})
policy_table=pd.DataFrame(policy_rows)
display(policy_table)
print("Policy measurements describe calibration data, not independent test performance.")
# Label-free covariate-shift diagnostics, with bins fixed from fitting data.
def covariate_drift(reference,current):
    rows=[]
    for column in NUMERIC:
        ref=reference[column].dropna().to_numpy(dtype="float64")
        cur=current[column].dropna().to_numpy(dtype="float64")
        if not len(ref) or not len(cur):
            continue
        cuts=np.unique(np.quantile(ref,np.linspace(0,1,11)[1:-1]))
        bins=np.r_[-np.inf,cuts,np.inf]
        a=np.histogram(ref,bins=bins)[0].astype(float)+.5
        b=np.histogram(cur,bins=bins)[0].astype(float)+.5
        a/=a.sum();b/=b.sum()
        rows.append({"feature":column,"PSI":float(np.sum((b-a)*np.log(b/a)))})
    return pd.DataFrame(rows).sort_values("PSI",ascending=False)
policy_drift=covariate_drift(X_train,X_policy)
display(policy_drift.head(10))
experiment_config={"version":EXPERIMENT_VERSION,"seed":SEED,"taxonomy":CLASSES,"winner":winner,
    "target_normal_group_FPR":TARGET_FPR,"policy_confidence":POLICY_CONFIDENCE,
    "policy_unit":"max score over Normal-labelled rows per related-feature group",
    "revision":REVISION,"cv_folds":CV_FOLDS,"group_significant_digits":4,
    "grouping_features":"core numeric measurements plus protocol/service/state",
    "CV_refit":"mean macro-F1 minus fold std (heuristic)",
    "development_folds":{"fitting":[0,1,2],"model_selection":[3],"policy_calibration":[4]},
    "run_neural":RUN_NEURAL,"neural_seeds":[SEED,SEED+7] if RUN_NEURAL else [],
    "neural_weighting":"normalized square-root inverse frequency",
    "neural_early_stopping":"validation macro-F1",
    "max_epochs":EPOCHS,"strategy":type(strategy).__name__ if RUN_NEURAL else "classical CPU",
    "historical_test_enabled":RUN_HISTORICAL_TEST,"fresh_holdout_available":False}


## 11. Optional validation feature sensitivity
Permutation importance describes predictive dependence, not causality. Compute it on validation only for the best CV classical candidate. It is optional because shuffling a flow feature can create unrealistic combinations. The core-feature comparison is already predeclared. Do not redesign features after looking at final-test errors and still call that test untouched.

In [ ]:
if RUN_PERMUTATION_IMPORTANCE:
    classical=cv_table.iloc[0]["model"]
    influence=permutation_importance(models[classical],X_val,y_val,
        scoring="f1_macro",n_repeats=3,random_state=SEED,n_jobs=N_JOBS)
    table=pd.DataFrame({"feature":PREDICTORS,"mean_drop":influence.importances_mean,
                        "std":influence.importances_std}).sort_values("mean_drop",ascending=False)
    display(table.head(12))
else:
    print("Validation permutation importance disabled.")

## 12. Optional historical benchmark replay
The published test results were inspected in the previous run. It cannot become untouched again by resetting Colab or choosing another random seed. **Default: `RUN_HISTORICAL_TEST=False`**. Run all completes development evidence without fetching that partition.

Enable replay only for learning diagnostics; exports label it historical and non-independent. A session guard blocks a repeat fetch. Scores on this filtered population must not be used to claim an independent improvement. A genuinely new compatible capture, with verified provenance, groups and integrity, is still required for final validation. Do not fabricate one or relabel development data as external testing.

In [ ]:
if RUN_HISTORICAL_TEST:
    if globals().get("_section2_test_evaluated",False):
        raise RuntimeError("Section 2 test already evaluated. Further development needs a fresh holdout.")
    _section2_test_evaluated=True
    raw_test,download_test=load_partition("test")
    test_pool,cleaning_test=clean_partition(raw_test)
    del raw_test
    test_pool["group"]=rounded_feature_groups(test_pool)
    exact_overlap=test_pool["feature_hash"].isin(train_pool["feature_hash"])
    group_overlap=test_pool["group"].isin(train_pool["group"])
    overlap_audit={"exact_overlap_rows":int(exact_overlap.sum()),
        "rounded_group_overlap_rows":int(group_overlap.sum()),
        "total_overlap_rows_removed":int((exact_overlap|group_overlap).sum())}
    testing=test_pool.loc[~(exact_overlap|group_overlap)].reset_index(drop=True)
    print("Test cleaning:",{k:v for k,v in cleaning_test.items() if isinstance(v,int)})
    print("Overlap filtering:",overlap_audit,"retained test rows:",len(testing))
    if len(testing)==0 or not (testing["target"]==NORMAL_ID).any() or not (testing["target"]!=NORMAL_ID).any():
        raise ValueError("Novelty test needs both Normal and attack-labelled rows.")
    assert not set(testing["group"]) & set(train_pool["group"])
    X_test,y_test=testing[PREDICTORS],testing["target"].to_numpy()
    binary_test=(y_test!=NORMAL_ID).astype(int)
    g_test=testing["group"].to_numpy()
    support=np.bincount(y_test,minlength=len(CLASSES))
    display(pd.DataFrame({"category":CLASSES,"retained_test_support":support}))
    if np.any(support==0):
        print("Some categories are absent after filtering. Macro-F1 uses the fixed taxonomy; AP/ROC omit undefined class curves.")
    unseen_rates={column:float((~X_test[column].isin(set(X_train[column].dropna()))).mean())
                  for column in CATEGORICAL}
    print("Test unseen-or-missing categorical fractions:",unseen_rates)
else:
    print("Historical test replay disabled. No fresh holdout result is claimed.")


## 13. Historical category and alert diagnostics (optional)
If enabled, report all frozen candidates, per-class support, category metrics and separate alert workload on the historical novelty subset. Retain the winner selected before policy calibration. These diagnostics can reveal shift but cannot validate a new performance claim. The policy target is group-level under stated assumptions; row FPR is separately reported.

In [ ]:
def category_metrics(y,probabilities):
    predicted=probabilities.argmax(axis=1)
    ap,roc=[],[]
    for i in range(len(CLASSES)):
        indicator=(y==i).astype(int)
        if indicator.min()!=indicator.max():
            ap.append(average_precision_score(indicator,probabilities[:,i]))
            roc.append(roc_auc_score(indicator,probabilities[:,i]))
    return {"accuracy":accuracy_score(y,predicted),
        "macro_F1":f1_score(y,predicted,labels=np.arange(len(CLASSES)),average="macro",zero_division=0),
        "macro_recall":recall_score(y,predicted,labels=np.arange(len(CLASSES)),average="macro",zero_division=0),
        "weighted_F1":f1_score(y,predicted,labels=np.arange(len(CLASSES)),average="weighted",zero_division=0),
        "macro_OVR_AP":float(np.mean(ap)) if ap else np.nan,
        "macro_OVR_ROC_AUC":float(np.mean(roc)) if roc else np.nan,
        "defined_OVR_classes":len(ap)}

if RUN_HISTORICAL_TEST:
    test_probabilities,test_rows={},[]
    for name,model in models.items():
        start=time.perf_counter()
        probabilities=predict_category(name,X_test)
        elapsed=time.perf_counter()-start
        test_probabilities[name]=probabilities
        scores=1-probabilities[:,NORMAL_ID]
        alerts=scores>=thresholds[name]
        normal_group_scores=pd.Series(scores[y_test==NORMAL_ID]).groupby(g_test[y_test==NORMAL_ID]).max().to_numpy()
        tn,fp,fn,tp=confusion_matrix(binary_test,alerts,labels=[0,1]).ravel()
        test_rows.append({"model":name,"selected_on_validation":name==winner,
            **category_metrics(y_test,probabilities),
            "alert_AP":average_precision_score(binary_test,scores),
            "alert_precision":precision_score(binary_test,alerts,zero_division=0),
            "alert_recall":recall_score(binary_test,alerts,zero_division=0),
            "alert_FPR":fp/max(1,fp+tn),
            "normal_group_alert_FPR":float(np.mean(normal_group_scores>=thresholds[name])),"alerts_per_1000_Normal":1000*fp/max(1,fp+tn),
            "alert_TN":int(tn),"alert_FP":int(fp),"alert_FN":int(fn),"alert_TP":int(tp),
            "fit_seconds":fit_seconds[name],"test_seconds":elapsed})
    test_table=pd.DataFrame(test_rows)
    display(test_table)
    winner_probabilities=test_probabilities[winner]
    winner_predictions=winner_probabilities.argmax(axis=1)
    per_class=pd.DataFrame(classification_report(y_test,winner_predictions,
        labels=np.arange(len(CLASSES)),target_names=CLASSES,output_dict=True,zero_division=0)).T
    display(per_class)
    fig,ax=plt.subplots(figsize=(10,8))
    ConfusionMatrixDisplay.from_predictions(y_test,winner_predictions,
        labels=np.arange(len(CLASSES)),display_labels=CLASSES,normalize="true",
        xticks_rotation=60,values_format=".2f",ax=ax,colorbar=False)
    ax.set_title("Validation winner: category recall by true class")
    plt.tight_layout();plt.show()
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    for name,probabilities in test_probabilities.items():
        scores=1-probabilities[:,NORMAL_ID]
        PrecisionRecallDisplay.from_predictions(binary_test,scores,name=name,ax=axes[0])
        RocCurveDisplay.from_predictions(binary_test,scores,name=name,ax=axes[1])
    for ax in axes:ax.legend(fontsize=7)
    plt.tight_layout();plt.show()
    fig,ax=plt.subplots(figsize=(8,5))
    for i,name in enumerate(CLASSES):
        if support[i]>0 and support[i]<len(y_test):
            PrecisionRecallDisplay.from_predictions((y_test==i).astype(int),
                winner_probabilities[:,i],name=name,ax=ax)
    ax.set_title("Selected candidate: one-vs-rest category PR");ax.legend(fontsize=7)
    plt.tight_layout();plt.show()
    if test_table.loc[test_table["model"]==winner,"alert_FPR"].iloc[0]>TARGET_FPR:
        print("Historical row FPR exceeds 1%. The policy calibrates group risk under assumptions; neither guarantees deployment row FPR. Do not retune on this benchmark.")
else:
    print("Historical test replay disabled. No fresh holdout result is claimed.")


## 14. Historical group bootstrap and aggregate errors (optional)
Resample entire approximate content groups and keep fixed taxonomy metrics. Intervals describe uncertainty within the historical filtered population, not the validity of a fresh holdout or immunity to deployment shift. Protocol slices report aggregate counts without raw flows.

In [ ]:
def group_intervals(y,probabilities,groups,threshold,repeats=BOOTSTRAPS):
    rng=np.random.default_rng(SEED+100)
    unique=np.unique(groups)
    # Build group indexes once, avoiding an O(groups * rows) scan.
    order=np.argsort(groups,kind="stable")
    boundaries=np.r_[0,np.flatnonzero(np.diff(groups[order]))+1,len(groups)]
    members=[order[boundaries[i]:boundaries[i+1]] for i in range(len(unique))]
    estimates=[]
    for _ in range(repeats):
        picks=rng.integers(0,len(unique),size=len(unique))
        idx=np.concatenate([members[i] for i in picks])
        yy,pp=y[idx],probabilities[idx]
        binary=(yy!=NORMAL_ID).astype(int)
        if len(np.unique(binary))<2:
            continue
        predicted=pp.argmax(axis=1)
        alerts=(1-pp[:,NORMAL_ID])>=threshold
        estimates.append([
            f1_score(yy,predicted,labels=np.arange(len(CLASSES)),average="macro",zero_division=0),
            recall_score(binary,alerts,zero_division=0),
            float(np.mean(alerts[binary==0]))])
    if not estimates:
        raise ValueError("No usable bootstrap replicates.")
    intervals=np.quantile(np.asarray(estimates),[.025,.975],axis=0)
    return pd.DataFrame({"metric":["macro_F1","alert_recall","alert_FPR"],
        "lower_95":intervals[0],"upper_95":intervals[1]}),len(estimates)

if RUN_HISTORICAL_TEST:
    uncertainty,valid_bootstraps=group_intervals(y_test,winner_probabilities,g_test,thresholds[winner])
    display(uncertainty)
    slices=pd.DataFrame({"protocol":X_test["proto"].fillna("missing").to_numpy(),
        "error":winner_predictions!=y_test,"attack":binary_test.astype(bool)})
    error_slices=slices.groupby("protocol").agg(count=("error","size"),
        category_error_rate=("error","mean"),attack_fraction=("attack","mean"))
    display(error_slices.loc[error_slices["count"]>=20].sort_values("category_error_rate",ascending=False))
    print("Valid bootstrap replicates:",valid_bootstraps,"— no raw records displayed.")
else:
    print("Historical test replay disabled. No fresh holdout result is claimed.")


## 15. Runtime-only evidence and experiment status
Always export CV, model-selection per-class results, policy calibration, covariate diagnostics, configuration, cleaning counts and all three partition fingerprints. Optional replay files explicitly say historical. No rows, datasets, fitted models or credentials are exported. Exports stay in Colab; inspect before sharing.

In [ ]:
output_dir=Path("/content/section2_results/v2_development")
output_dir.mkdir(parents=True,exist_ok=True)
cv_table.assign(parameters=cv_table["parameters"].map(json.dumps)).to_csv(output_dir/"classical_cv.csv",index=False)
selection_table.to_csv(output_dir/"model_selection.csv",index=False)
selection_per_class.to_csv(output_dir/"selection_per_class.csv")
policy_table.to_csv(output_dir/"policy_calibration.csv",index=False)
policy_drift.to_csv(output_dir/"policy_covariate_drift.csv",index=False)
def fingerprint(frame):
    return hashlib.sha256("\n".join(sorted(frame["feature_hash"].astype(str))).encode()).hexdigest()
experiment_config["historical_test_enabled"]=bool(RUN_HISTORICAL_TEST)
evidence={"configuration":experiment_config,"software":versions,"download":{"train":download_train},
    "cleaning":{"train":cleaning_train},
    "partition_sizes":{name:len(frame) for name,frame in development_parts.items()},
    "partition_fingerprints":{name:fingerprint(frame) for name,frame in development_parts.items()},
    "alert_thresholds":thresholds,"policy_details":policy_details,
    "evaluation_status":"development_only_no_fresh_holdout",
    "limitations":["No verified session/time independence","Historical synthetic attack benchmark",
        "Approximate group independence and exchangeability unverified",
        "Normal-group threshold is not a row-FPR or deployment guarantee",
        "No new independent test data","No deployment validation"]}
if RUN_HISTORICAL_TEST:
    test_table.to_csv(output_dir/"historical_test_metrics.csv",index=False)
    per_class.to_csv(output_dir/"historical_winner_per_class.csv")
    uncertainty.to_csv(output_dir/"historical_winner_intervals.csv",index=False)
    evidence.update({"historical_download":download_test,"historical_cleaning":cleaning_test,
        "historical_overlap":overlap_audit,"historical_test_fingerprint":fingerprint(testing),
        "historical_test_rows":len(testing),"valid_bootstraps":valid_bootstraps,
        "historical_test_is_independent":False})
(output_dir/"experiment_evidence.json").write_text(json.dumps(evidence,indent=2,allow_nan=False),encoding="utf-8")
print("Aggregate development evidence saved in Colab runtime:",output_dir)
print("No new independent test performance is claimed.")


## 16. Learning conclusion
Explain cleaning exclusions and class support; train/CV gaps; randomized-tree and core-feature comparisons; neural macro-F1 curves; and the calibration tradeoff between missed attacks and Normal-group false alerts. Report reject-all policies honestly.

Separate model-selection results, policy-calibration measurements and optional historical diagnostics. No new independent test scores are claimed. Keep the old run as historical evidence rather than comparing new development scores as though they were the same experiment. Further validation needs a new independent compatible capture.

References: [UNSW-NB15](https://research.unsw.edu.au/projects/unsw-nb15-dataset), [grouped CV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedGroupKFold.html), [leakage guidance](https://scikit-learn.org/stable/common_pitfalls.html), [Extra Trees](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesClassifier.html), [TensorFlow strategies](https://www.tensorflow.org/guide/distributed_training).